# 06 · Figures and tables

Builds every paper figure (vector PDF + 200 dpi PNG) and every LaTeX / Markdown table from what the earlier notebooks wrote to reports/. Anything whose inputs are missing is skipped with a note. The RQ3a system benchmark and the scenario runs exist only after a LOCAL run (broker, device fleet); import them with scripts.import_artifacts and re-run this notebook.

*Estimated time: about 1 min.*  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/furkantopcuu/airq-iot/blob/main/notebooks/06_figures_tables.ipynb)

In [ ]:
# --- parameters -------------------------------------------------------------------------
QUICK = True        # 2 stations, short run (< 5 min); set False for the full 12-station run
USE_DRIVE = False   # Colab only: mount Google Drive to keep results between sessions
RUN_OPTUNA = False  # hyper-parameter search (notebook 02 onwards)
SEEDS = [0]


In [ ]:
# --- setup (runs on Colab, skipped locally) ---------------------------------------------
import os
import subprocess
import sys
IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/furkantopcuu/airq-iot.git"
if IN_COLAB:
    if not os.path.exists("/content/airq-iot"):
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/airq-iot"], check=True)
    os.chdir("/content/airq-iot")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml]"], check=True)
    sys.path.insert(0, "/content/airq-iot/src")  # no restart needed after the editable install


In [ ]:
from airq.config import load_config
from airq.env import mount_drive_if_wanted
from airq.pipelines.colab import env_summary

mount_drive_if_wanted(USE_DRIVE)
cfg = load_config()
env_summary(cfg)


In [ ]:
from IPython.display import Image, display
from airq.pipelines.figures import make_all

made, skipped = make_all(cfg)
for stage, why in skipped.items():
    print(f"skipped {stage}: {why}")
for p in made:
    if p.suffix == ".png":
        display(Image(filename=str(p)))


In [ ]:
from IPython.display import Markdown, display
from airq.env import resolve_home
from airq.pipelines.tables import build_all

made_tables = build_all(cfg)  # paper/tables/*.tex and *.md from reports/tables/*.csv
for name, files in made_tables.items():
    display(Markdown(f"**{name}**\n\n" + files[1].read_text(encoding="utf-8")))
print("LaTeX tables in", resolve_home() / "paper" / "tables")


In [ ]:
# --- export: reports/ and models/ -> airq_artifacts_<date>.zip --------------------------
from airq.pipelines.colab import export_artifacts

zip_path = export_artifacts(cfg)
print("Wrote", zip_path.name)
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
# locally:  python -m scripts.import_artifacts <zip>   (also runs the manifest check)
